In [ ]:
!pip install ultralytics

In [ ]:

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import xml.etree.ElementTree as ET
import glob
import yaml
import json
import random
from ultralytics import YOLO
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ===== COCO DATASET DOWNLOAD =====
print("=== Downloading COCO Dataset ===")

# Create directories
!mkdir -p ./coco/images
!mkdir -p ./coco/annotations

# Download COCO images and annotations
!wget http://images.cocodataset.org/zips/train2017.zip
!wget http://images.cocodataset.org/zips/val2017.zip
!wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip

# Extract files
!unzip train2017.zip -d ./coco/images/
!unzip val2017.zip -d ./coco/images/
!unzip annotations_trainval2017.zip -d ./coco/annotations/

print("COCO Dataset downloaded and extracted successfully!")

In [ ]:
# ===== PASCAL VOC DATASET DOWNLOAD (Second Dataset) =====
print("\n=== Downloading PASCAL VOC 2007 Dataset ===")

# Create directory for VOC data
!mkdir -p VOCdevkit

# Download the VOC 2007 test set
!wget http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtest_06-Nov-2007.tar

# Extract the downloaded tar file
!tar -xf VOCtest_06-Nov-2007.tar -C VOCdevkit/

# Verify files
!ls VOCdevkit/VOC2007/

print("PASCAL VOC 2007 Dataset downloaded and extracted successfully!")

In [ ]:
# Load YOLOv8 model (pre-trained)
print("\n=== Loading YOLOv8 Model ===")
model = YOLO('yolov8n.pt')  # YOLOv8 Nano - fastest version
print("YOLOv8 model loaded successfully!")

# ===== INITIAL PREDICTIONS ON COCO VALIDATION SET =====
print("\n=== Running Initial Predictions on COCO Dataset ===")
results = model.predict(source='/content/coco/images/val2017', save=True, conf=0.25)

# Display first 3 predictions
print("Displaying sample predictions...")
for i, r in enumerate(results[:3]):
    r.show()
    print(f"Sample {i+1} prediction displayed")

In [ ]:
# ===== INITIAL COCO EVALUATION =====
print("\n=== Initial COCO Dataset Evaluation ===")
coco_metrics = model.val(data='coco.yaml', imgsz=640)
print(f"mAP@0.5: {coco_metrics.box.map50:.3f}")
print(f"mAP@0.5:0.95: {coco_metrics.box.map:.3f}")
print(f"Precision: {coco_metrics.box.mp:.3f}")
print(f"Recall: {coco_metrics.box.mr:.3f}")

In [ ]:
def convert_voc_to_yolo():
    """
    Convert PASCAL VOC XML annotations to YOLO format
    """
    print("\n=== Converting PASCAL VOC to YOLO Format ===")
    
    # VOC class names (20 classes)
    voc_classes = ['aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car',
                   'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike',
                   'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor']

    # Create YOLO dataset directory
    yolo_path = Path('/content/VOC2007_YOLO')
    yolo_path.mkdir(exist_ok=True)
    (yolo_path / 'images').mkdir(exist_ok=True)
    (yolo_path / 'labels').mkdir(exist_ok=True)

    # Convert XML annotations to YOLO format
    xml_files = glob.glob('/content/VOCdevkit/VOC2007/Annotations/*.xml')
    print(f"Converting {len(xml_files)} annotation files...")

    for xml_file in xml_files:
        tree = ET.parse(xml_file)
        root = tree.getroot()

        # Get image dimensions
        size = root.find('size')
        width = int(size.find('width').text)
        height = int(size.find('height').text)

        # Create YOLO annotation file
        base_name = Path(xml_file).stem
        yolo_file = yolo_path / 'labels' / f"{base_name}.txt"

        with open(yolo_file, 'w') as f:
            for obj in root.findall('object'):
                class_name = obj.find('name').text
                if class_name in voc_classes:
                    class_id = voc_classes.index(class_name)

                    # Get bounding box
                    bbox = obj.find('bndbox')
                    xmin = float(bbox.find('xmin').text)
                    xmax = float(bbox.find('xmax').text)
                    ymin = float(bbox.find('ymin').text)
                    ymax = float(bbox.find('ymax').text)

                    # Convert to YOLO format (normalized)
                    x_center = (xmin + xmax) / 2.0 / width
                    y_center = (ymin + ymax) / 2.0 / height
                    w = (xmax - xmin) / width
                    h = (ymax - ymin) / height

                    f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}\n")

        # Copy image file
        filename = root.find('filename').text
        img_src = f"/content/VOCdevkit/VOC2007/JPEGImages/{filename}"
        img_dst = yolo_path / 'images' / filename
        if os.path.exists(img_src):
            os.system(f'cp "{img_src}" "{img_dst}"')

    print("PASCAL VOC to YOLO conversion completed!")
    return str(yolo_path)

In [ ]:
# Run conversion
yolo_dataset_path = convert_voc_to_yolo()

# Create YOLO dataset configuration
dataset_config = {
    'path': yolo_dataset_path,
    'train': 'images',
    'val': 'images',
    'test': 'images',
    'nc': 20,
    'names': ['aeroplane', 'bicycle', 'bird', 'boat', 'bottle', 'bus', 'car',
              'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse', 'motorbike',
              'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor']
}

# Save config file
config_path = '/content/voc2007_yolo.yaml'
with open(config_path, 'w') as f:
    yaml.dump(dataset_config, f)

print(f"Dataset config saved: {config_path}")

In [ ]:
# ===== PASCAL VOC EVALUATION =====
print("\n=== PASCAL VOC Dataset Evaluation ===")
voc_metrics = model.val(
    data=config_path,
    imgsz=640,
    conf=0.001,
    iou=0.6,
    verbose=True
)

print(f"mAP@0.5:     {voc_metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {voc_metrics.box.map:.4f}")
print(f"Precision:   {voc_metrics.box.mp:.4f}")
print(f"Recall:      {voc_metrics.box.mr:.4f}")

In [ ]:
def calculate_iou(box1, box2):
    """
    Calculate Intersection over Union (IoU) for two bounding boxes
    box format: [x1, y1, x2, y2]
    
    Args:
        box1: First bounding box [x1, y1, x2, y2]
        box2: Second bounding box [x1, y1, x2, y2]
    
    Returns:
        float: IoU value between 0 and 1
    """
    # Calculate intersection coordinates
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    
    # Check if there's no intersection
    if x2 <= x1 or y2 <= y1:
        return 0.0
    
    # Calculate intersection area
    intersection = (x2 - x1) * (y2 - y1)
    
    # Calculate areas of both boxes
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    # Calculate union
    union = area1 + area2 - intersection
    
    # Return IoU
    return intersection / union if union > 0 else 0.0


In [ ]:
def evaluate_iou_on_sample(model, image_path, ground_truth_boxes):
    """
    Calculate average IOU for a sample image
    
    Args:
        model: YOLO model
        image_path: Path to image
        ground_truth_boxes: List of ground truth boxes [[x1,y1,x2,y2], ...]
    
    Returns:
        float: Average IoU score
    """
    results = model.predict(source=image_path, conf=0.25, verbose=False)
    
    if not results or len(results[0].boxes) == 0:
        return 0.0
    
    predicted_boxes = results[0].boxes.xyxy.cpu().numpy()  # [x1, y1, x2, y2]
    
    ious = []
    for pred_box in predicted_boxes:
        best_iou = 0.0
        for gt_box in ground_truth_boxes:
            iou = calculate_iou(pred_box, gt_box)
            best_iou = max(best_iou, iou)
        ious.append(best_iou)
    
    return np.mean(ious) if ious else 0.0


In [ ]:
# Test IoU calculation
print("\n=== Testing IoU Calculation ===")
# Example boxes for testing
test_box1 = [100, 100, 200, 200]  # Perfect overlap
test_box2 = [100, 100, 200, 200]
test_box3 = [150, 150, 250, 250]  # Partial overlap
test_box4 = [300, 300, 400, 400]  # No overlap

print(f"IoU (identical boxes): {calculate_iou(test_box1, test_box2):.3f}")
print(f"IoU (overlapping boxes): {calculate_iou(test_box1, test_box3):.3f}")
print(f"IoU (non-overlapping boxes): {calculate_iou(test_box1, test_box4):.3f}")

In [ ]:
def analyze_detection_cases(model, image_folder, num_samples=15):
    """
    Analyze success and failure cases for YOLO model
    
    Args:
        model: YOLO model
        image_folder: Folder containing images
        num_samples: Number of images to analyze
    
    Returns:
        tuple: (success_cases, failure_cases)
    """
    image_files = glob.glob(f"{image_folder}/*.jpg")
    if len(image_files) > num_samples:
        image_files = random.sample(image_files, num_samples)
    
    success_cases = []
    failure_cases = []
    
    print(f"\n=== ANALYZING {len(image_files)} DETECTION CASES ===")
    
    for i, img_path in enumerate(image_files):
        print(f"Analyzing image {i+1}/{len(image_files)}: {os.path.basename(img_path)}")
        
        results = model.predict(source=img_path, conf=0.25, verbose=False)
        
        if not results:
            continue
            
        result = results[0]
        num_detections = len(result.boxes) if result.boxes is not None else 0
        img_name = os.path.basename(img_path)
        
        if num_detections > 0:
            # Get confidence scores and classes
            confidences = result.boxes.conf.cpu().numpy() if result.boxes is not None else []
            avg_confidence = np.mean(confidences) if len(confidences) > 0 else 0
            classes = result.boxes.cls.cpu().numpy().tolist() if result.boxes is not None else []
            class_names = [model.names[int(cls)] for cls in classes]
            
            # Success criteria: reasonable number of detections with good confidence
            if avg_confidence > 0.4 and num_detections <= 10:  # Not too many false positives
                success_cases.append({
                    'image': img_name,
                    'detections': num_detections,
                    'avg_confidence': avg_confidence,
                    'max_confidence': np.max(confidences) if len(confidences) > 0 else 0,
                    'classes': class_names,
                    'reason': 'High confidence detections'
                })
            else:
                failure_cases.append({
                    'image': img_name,
                    'detections': num_detections,
                    'avg_confidence': avg_confidence,
                    'classes': class_names,
                    'issue': 'Low confidence detections' if avg_confidence <= 0.4 else 'Too many detections (possible false positives)'
                })
        else:
            failure_cases.append({
                'image': img_name,
                'detections': 0,
                'avg_confidence': 0,
                'classes': [],
                'issue': 'No objects detected'
            })
    
    return success_cases, failure_cases

In [ ]:
def print_case_analysis(success_cases, failure_cases):
    """
    Print detailed analysis of success and failure cases
    """
    print(f"\n=== SUCCESS CASES ANALYSIS ({len(success_cases)} cases) ===")
    for i, case in enumerate(success_cases[:5]):  # Show top 5
        print(f"\n{i+1}. Image: {case['image']}")
        print(f"   Detections: {case['detections']}")
        print(f"   Avg Confidence: {case['avg_confidence']:.3f}")
        print(f"   Max Confidence: {case['max_confidence']:.3f}")
        print(f"   Objects: {', '.join(case['classes'])}")
        print(f"   Reason: {case['reason']}")
    
    print(f"\n=== FAILURE CASES ANALYSIS ({len(failure_cases)} cases) ===")
    for i, case in enumerate(failure_cases[:5]):  # Show top 5
        print(f"\n{i+1}. Image: {case['image']}")
        print(f"   Issue: {case['issue']}")
        print(f"   Detections: {case['detections']}")
        print(f"   Avg Confidence: {case['avg_confidence']:.3f}")
        if case['classes']:
            print(f"   Objects detected: {', '.join(case['classes'])}")
    
    # Statistical analysis
    print(f"\n=== STATISTICAL SUMMARY ===")
    total_cases = len(success_cases) + len(failure_cases)
    success_rate = len(success_cases) / total_cases * 100 if total_cases > 0 else 0
    
    print(f"Total analyzed cases: {total_cases}")
    print(f"Success cases: {len(success_cases)} ({success_rate:.1f}%)")
    print(f"Failure cases: {len(failure_cases)} ({100-success_rate:.1f}%)")
    
    # Analyze failure reasons
    failure_reasons = {}
    for case in failure_cases:
        reason = case['issue']
        failure_reasons[reason] = failure_reasons.get(reason, 0) + 1
    
    print("\nFailure breakdown:")
    for reason, count in failure_reasons.items():
        print(f"  - {reason}: {count} cases")


In [ ]:
def analyze_yolo_strengths_weaknesses(success_cases, failure_cases):
    """
    Analyze YOLO model strengths and weaknesses based on test cases
    """
    print(f"\n=== YOLO MODEL ANALYSIS ===")
    
    # Analyze successful detections
    successful_classes = []
    for case in success_cases:
        successful_classes.extend(case['classes'])
    
    # Count class occurrences
    class_counts = {}
    for cls in successful_classes:
        class_counts[cls] = class_counts.get(cls, 0) + 1
    
    print("\n YOLO STRENGTHS:")
    print("1. Speed and Efficiency:")
    print("   - Single-stage detector provides fast inference")
    print("   - Real-time detection capability")
    
    print("2. Good Performance on Common Objects:")
    if class_counts:
        top_classes = sorted(class_counts.items(), key=lambda x: x[1], reverse=True)[:3]
        print(f"   - Most successfully detected: {', '.join([cls for cls, count in top_classes])}")
    
    print("3. End-to-End Learning:")
    print("   - Direct bounding box and class prediction")
    print("   - No need for region proposals")
    
    print("4. Multi-Scale Detection:")
    print("   - Can detect objects of various sizes")
    
    print("\n YOLO WEAKNESSES:")
    print("1. Small Object Detection:")
    print("   - May struggle with very small objects")
    print("   - Limited by grid cell resolution")
    
    print("2. Crowded Scenes:")
    print("   - May miss objects in densely packed scenes")
    print("   - Grid cell limitation affects nearby objects")
    
    print("3. Aspect Ratio Sensitivity:")
    print("   - May struggle with objects having unusual aspect ratios")
    
    print("4. False Positive Issues:")
    false_positive_cases = [case for case in failure_cases if 'false positive' in case.get('issue', '').lower()]
    if false_positive_cases:
        print(f"   - Detected {len(false_positive_cases)} cases with potential false positives")

In [ ]:
def visualize_detection_results(model, image_path, save_path=None):
    """
    Visualize detection results with bounding boxes and labels
    """
    if not os.path.exists(image_path):
        print(f"Image not found: {image_path}")
        return
    
    # Get predictions
    results = model.predict(source=image_path, conf=0.25, verbose=False)
    
    if not results:
        print("No results found")
        return
    
    result = results[0]
    
    # Create visualization
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    
    # Get image with drawn boxes
    im_array = result.plot()  # This returns the image with boxes drawn
    
    # Convert BGR to RGB for matplotlib
    im_rgb = cv2.cvtColor(im_array, cv2.COLOR_BGR2RGB)
    
    ax.imshow(im_rgb)
    ax.axis('off')
    ax.set_title(f'YOLO Detection Results - {os.path.basename(image_path)}', fontsize=14, fontweight='bold')
    
    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=150)
    
    plt.tight_layout()
    plt.show()
    
    # Print detection details
    if result.boxes is not None and len(result.boxes) > 0:
        print(f"\n Detection Summary for {os.path.basename(image_path)}:")
        print(f"Total objects detected: {len(result.boxes)}")
        print("\nDetailed results:")
        
        for i, (conf, cls) in enumerate(zip(result.boxes.conf, result.boxes.cls)):
            class_name = model.names[int(cls)]
            print(f"  {i+1}. {class_name}: {conf:.3f} confidence")
    else:
        print(f"No objects detected in {os.path.basename(image_path)}")

In [ ]:
def show_sample_detections(model, image_folder, num_samples=3):
    """
    Show sample detection results from a folder
    """
    image_files = glob.glob(f"{image_folder}/*.jpg")[:num_samples]
    
    print(f"\n=== SAMPLE DETECTION VISUALIZATIONS ===")
    for i, img_path in enumerate(image_files):
        print(f"\nSample {i+1}: {os.path.basename(img_path)}")
        visualize_detection_results(model, img_path)
# Analyze detection cases
success_cases, failure_cases = analyze_detection_cases(model, '/content/coco/images/val2017', num_samples=15)   
# Print case analysis
print_case_analysis(success_cases, failure_cases)   
# Analyze YOLO strengths and weaknesses
analyze_yolo_strengths_weaknesses(success_cases, failure_cases)
# Visualize detection results for a sample image
sample_image = '/content/coco/images/val2017/000000039769.jpg'
visualize_detection_results(model, sample_image, save_path='/content/sample_detection_result.png')
# Show sample detections from a folder
show_sample_detections(model, '/content/coco/images/val2017', num_samples=3)    
